# 03 · Running on IBM Quantum hardware

Plan: optimise the QAOA parameters on the noise-free simulator (free), then spend hardware time only on
sampling with those fixed parameters. Two `SamplerV2` jobs (with and without error suppression) plus one
`EstimatorV2` job with zero-noise extrapolation. Every job ID is logged in `results/hardware_jobs.jsonl`.

With `USE_REAL_HARDWARE = False` the notebook runs end to end on a local noisy model of an IBM Heron device.
The ZNE estimator step is skipped locally: the runtime's local mode measures all 156 device qubits for it,
which does not fit in a laptop simulator.

**Before the first real run**, save your credentials once (never commit them):

```python
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="<API key>", instance="<CRN>")
```

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
USE_REAL_HARDWARE = False   # set True for the real run
N, DENSITY, SEED, P = 8, 0.3, 1, 2
SHOTS = 4096
FAKE_DEVICE = "FakeFez"     # local noisy model of a Heron r2 device

## 1. Problem and simulator-optimised parameters

In [ ]:
from microgrid import (make_microgrid, build_qubo, prepare, brute_force, solve_layerwise, build_ansatz,
                       probability_of_optimal, circuit_stats, dispatch_report)

grid = make_microgrid(N, DENSITY, SEED)
problem = prepare(build_qubo(grid))
exact, land = brute_force(problem.qubo)
ideal = solve_layerwise(problem, max_reps=P)[-1]
print("ideal P(optimal):", round(probability_of_optimal(ideal.probs, land.optimal_indices), 4))
print("ideal <E>:", round(ideal.expectation, 3), " exact optimum:", round(exact.energy, 3))

## 2. Backend and transpilation

In [ ]:
from microgrid.hardware import fake_backend, noisy_simulator, real_backend, transpile_for

if USE_REAL_HARDWARE:
    backend = real_backend(min_qubits=N)
    target = backend
else:
    backend = noisy_simulator(FAKE_DEVICE)
    target = fake_backend(FAKE_DEVICE)
print("backend:", backend.name if hasattr(backend, "name") else backend)

sampling_ansatz = build_ansatz(problem.normalized_hamiltonian, P, measure=True)
energy_ansatz = build_ansatz(problem.normalized_hamiltonian, P)
isa_sampling = transpile_for(sampling_ansatz.circuit, target)
isa_energy = transpile_for(energy_ansatz.circuit, target)
print("sampling circuit:", circuit_stats(isa_sampling))

## 3. Sampling with and without error suppression

In [ ]:
from qiskit_ibm_runtime import Batch, SamplerV2
from microgrid import SamplerEvaluator
from microgrid.hardware import mitigated_sampler, zne_estimator, log_job

def run_all(mode):
    tag = f"n{N}_d{DENSITY}_s{SEED}_p{P}"
    on_job = (lambda kind: (lambda job: log_job(job, f"{kind}_{tag}"))) if USE_REAL_HARDWARE else (lambda kind: None)
    raw = SamplerEvaluator(SamplerV2(mode=mode), sampling_ansatz, isa_sampling, SHOTS, on_job=on_job("sampler_raw"))
    sup = SamplerEvaluator(mitigated_sampler(mode, SHOTS), sampling_ansatz, isa_sampling, SHOTS,
                           on_job=on_job("sampler_dd_twirl"))
    out = {"raw": raw.probabilities(ideal.theta), "suppressed": sup.probabilities(ideal.theta)}

    # Zero-noise extrapolation runs on real hardware only. In local fake-backend mode the runtime
    # Estimator measures every device qubit (156 on Heron), which no laptop simulator can hold.
    out["zne_energy"] = None
    if USE_REAL_HARDWARE:
        estimator = zne_estimator(mode)
        observable = problem.hamiltonian.apply_layout(isa_energy.layout)
        job = estimator.run([(isa_energy, observable, energy_ansatz.values(ideal.theta, isa_energy))])
        log_job(job, f"estimator_zne_{tag}")
        out["zne_energy"] = float(job.result()[0].data.evs) + problem.offset
    return out

if USE_REAL_HARDWARE:
    with Batch(backend=backend) as batch:
        runs = run_all(batch)
else:
    runs = run_all(backend)

## 4. Results

In [ ]:
p_random = len(land.optimal_indices) / 2**N
rows = [("ideal simulator", ideal.probs)] + [(f"{'hardware' if USE_REAL_HARDWARE else 'noisy model'} ({k})", v)
                                             for k, v in runs.items() if k != "zne_energy"]
for label, probs in rows:
    p_opt = probability_of_optimal(probs, land.optimal_indices)
    sampled = np.flatnonzero(probs)
    best = sampled[np.argmin(problem.energies[sampled])]
    print(f"{label:32s} <E> ratio={land.approximation_ratio(probs @ problem.energies):.3f}  "
          f"P(opt)={p_opt:.4f} ({p_opt / p_random:5.1f}x random)  best sample optimal: "
          f"{np.isclose(problem.energies[best], exact.energy)}")
if runs["zne_energy"] is not None:
    print(f"\n<E> ideal = {ideal.expectation:.3f} | ZNE-mitigated estimator <E> = {runs['zne_energy']:.3f}")

In [ ]:
import json

record = {
    "backend": backend.name if hasattr(backend, "name") else str(backend),
    "real_hardware": USE_REAL_HARDWARE,
    "instance": {"n": N, "density": DENSITY, "seed": SEED, "p": P, "shots": SHOTS},
    "exact_energy": exact.energy,
    "ideal": {"expectation": ideal.expectation,
              "p_opt": probability_of_optimal(ideal.probs, land.optimal_indices)},
    "runs": {k: {"expectation": float(v @ problem.energies),
                 "p_opt": probability_of_optimal(v, land.optimal_indices)}
             for k, v in runs.items() if k != "zne_energy"},
    "zne_energy": runs["zne_energy"],
    "circuit": circuit_stats(isa_sampling),
}
name = "hardware_results.json" if USE_REAL_HARDWARE else "noisy_model_results.json"
(ROOT / "results" / name).write_text(json.dumps(record, indent=2), encoding="utf-8")
print("saved results/" + name)

**Reading the numbers honestly**

* On hardware, gate errors spread probability away from the optimum, so `P(opt)` falls towards the random-guess value as `p` and the number of `ZZ` terms grow.
* The decision is the best sample, so even a weakened distribution can still return the optimal dispatch at these sizes, but so do classical methods, faster.
* What this run demonstrates is the full pipeline on real hardware and how noise scales with grid density, not a quantum advantage.